# k-Vizinhos mais próximos (k-NN)

**Objetivo:** classificar o Iris com k-NN, ver como o número de vizinhos $k$ controla o compromisso viés–variância (a curva de acurácia por $k$) e desenhar a fronteira de decisão para dois valores de $k$.

In [ ]:
# bibliotecas base
import numpy as np
import pandas as pd

# Plotly para os gráficos (interativos e leves no Colab)
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
pio.templates.default = "simple_white"

# paleta do curso (a mesma do site)
AZUL, VERMELHO, VERDE = "#3266ad", "#c0392b", "#1a7a4a"
TINTA, SUAVE = "#1c1e15", "#6b7050"

# reprodutibilidade: uma única semente para tudo que é aleatório
SEMENTE = 42
np.random.seed(SEMENTE)

## 1. Dados e a importância de padronizar

O k-NN mede **distâncias**, então padronizamos as características (dentro de um `Pipeline`, para não vazar o teste). Usamos o Iris completo.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline

iris = load_iris()
X, y = iris.data, iris.target
print("X:", X.shape, "| classes:", list(iris.target_names))

## 2. A curva de acurácia por k

Para cada $k$ (ímpar, para evitar empates), medimos a acurácia por validação cruzada de 5 dobras. Um laço explícito, um $k$ por vez.

In [ ]:
ks = list(range(1, 40, 2))
acuracias = []
for k in ks:
    modelo = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k))
    ac = cross_val_score(modelo, X, y, cv=5).mean()
    acuracias.append(ac)
    print("k =", str(k).rjust(2), "-> acuracia CV =", round(ac, 3))
melhor_k = ks[int(np.argmax(acuracias))]
print("melhor k:", melhor_k)

In [ ]:
figura = go.Figure(go.Scatter(x=ks, y=acuracias, mode="lines+markers",
                              line=dict(color=AZUL)))
figura.add_vline(x=melhor_k, line_dash="dash", line_color=VERDE,
                 annotation_text=f"melhor k = {melhor_k}")
figura.update_layout(title="Acuracia (validacao cruzada) vs numero de vizinhos",
                     xaxis_title="k", yaxis_title="acuracia", height=360,
                     margin=dict(l=10, r=10, t=50, b=10))
figura.show()

## 3. A fronteira de decisão muda com k

Usando dois preditores (comprimento e largura da pétala), pintamos a região prevista para $k=1$ (recortada) e $k=25$ (suave).

In [ ]:
X2 = X[:, 2:4]   # petala: comprimento e largura
passo = 0.02
gx, gy = np.meshgrid(np.arange(X2[:, 0].min()-0.5, X2[:, 0].max()+0.5, passo),
                     np.arange(X2[:, 1].min()-0.5, X2[:, 1].max()+0.5, passo))
grade = np.c_[gx.ravel(), gy.ravel()]

from plotly.subplots import make_subplots
figura = make_subplots(rows=1, cols=2, subplot_titles=("k = 1", "k = 25"))
coluna = 1
for k in [1, 25]:
    modelo = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k)).fit(X2, y)
    zz = modelo.predict(grade).reshape(gx.shape)
    figura.add_trace(go.Heatmap(x=gx[0], y=gy[:, 0], z=zz, showscale=False,
                                colorscale="Blugrn", opacity=0.35), row=1, col=coluna)
    figura.add_trace(go.Scatter(x=X2[:, 0], y=X2[:, 1], mode="markers",
                                marker=dict(color=y, colorscale="Blugrn", size=6,
                                            line=dict(width=0.5, color="white")),
                                showlegend=False), row=1, col=coluna)
    coluna += 1
figura.update_layout(title="Fronteira de decisao: k=1 recortada, k=25 suave",
                     height=380, margin=dict(l=10, r=10, t=60, b=10))
figura.show()

## Exercício

Refaça a curva de acurácia **sem** o `StandardScaler` (troque o pipeline por um `KNeighborsClassifier` puro). No Iris o efeito é pequeno porque as escalas são parecidas — mas em que tipo de dado a padronização seria decisiva?

<details><summary>Ver resposta</summary>

Seria decisiva quando os preditores têm **escalas muito diferentes** — por exemplo, colesterol em mg/dL (centenas) misturado com uma proporção (0 a 1). Sem padronizar, a variável de valores grandes domina a distância euclidiana e as outras são praticamente ignoradas. No Iris, as quatro medidas estão todas em centímetros, então o impacto é pequeno.

</details>